In [1]:
# =============================================================================
# Q-LEARNING, SARSA, DQN — learn WHILE you walk (not only at the finish line)
# =============================================================================
#
# Last notebook (RL foundations):
#   Dynamic Programming → needs the full rulebook P (model-based planning)
#   Monte Carlo         → learns from COMPLETE episodes (wait until the end)
#
# This notebook: Temporal Difference (TD) methods.
#   Update your guess AFTER EACH STEP, using the reward you just got PLUS
#   your current guess of the next state's value. No rulebook. No waiting
#   for the episode to finish.
#
# Analogy — grading a hike as you go:
#   Monte Carlo: finish the trail, then score every mile from total time.
#   TD:         at each milepost, update "how good is this spot?" using
#               (time so far) + (my estimate of the remaining trail).
#
# That "estimate of the remaining trail" is bootstrapping — learning from
# your own current estimates. Powerful, a bit like standing on your own
# shoulders. Works online, step by step.
#


# -----------------------------------------------------------------------------
# 1. TEMPORAL DIFFERENCE (TD) LEARNING — the big idea
# -----------------------------------------------------------------------------
#
# Recall: return from time t is the discounted sum of future rewards:
#   G_t = R_{t+1} + γ R_{t+2} + γ² R_{t+3} + …
#
# Monte Carlo waits for the whole G_t, then averages.
#
# TD(0) for state values instead uses a ONE-STEP target:
#   target ≈ R_{t+1} + γ V(S_{t+1})     ← "reward now + guess of what comes next"
#
# Update:
#   V(S_t) ← V(S_t) + α [ target − V(S_t) ]
#
# The thing in brackets is the TD error δ:
#   δ = (what just happened + my guess for next) − (what I thought before)
#
# If δ > 0: "that step was BETTER than I expected → raise V(S_t)"
# If δ < 0: "worse than expected → lower V(S_t)"
#
# α (learning rate): how big a step toward the new target (e.g. 0.1).
# γ (discount): how much future rewards matter (same as before).
#
# For CONTROL (picking actions) we track Q(s,a) = "how good is action a in s?"
# Two famous recipes: SARSA and Q-learning. They differ in ONE word of the
# target — and that one word changes their personality.
#


# -----------------------------------------------------------------------------
# 2. SARSA — On-policy TD control  (State → Action → Reward → State → Action)
# -----------------------------------------------------------------------------
#
# Name comes from the tuple it uses each update:
#   (S, A, R, S', A')
#
# After taking A in S, seeing R and landing in S', SARSA ALSO picks the NEXT
# action A' the SAME way it usually acts (e.g. ε-greedy), then updates:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ Q(S', A') − Q(S,A) ]
#                              └─────┬─────┘
#                         value of the action we WILL take next
#
# On-policy = "learn about the policy you are actually following."
# Including the exploration! If ε-greedy sometimes walks off a cliff,
# SARSA learns "near the cliff, that Q is dangerous" because A' might be
# the clumsy exploratory step.
#
# Personality: cautious. Good when exploration is risky in the real world
# (robots, medicine). The learned policy matches "how I behave while learning."
#


# -----------------------------------------------------------------------------
# 3. Q-LEARNING — Off-policy TD control  (dream of the best next move)
# -----------------------------------------------------------------------------
#
# Same setup: take A in S (often still ε-greedy for exploration), see R, S'.
# But the TARGET pretends the NEXT action is the BEST one, not the one you
# might actually take:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ max_{a'} Q(S', a') − Q(S,A) ]
#                              └──────────┬──────────┘
#                         value of the BEST action available next
#
# Off-policy = "learn about a DIFFERENT (greedy) policy while behaving with
# exploration." Behavior policy explores; target policy is greedy.
#
# Personality: optimistic / bold. Learns the optimal path even if while
# training you sometimes take dumb exploratory moves. Classic cliff-walk
# demo: Q-learning hugs the cliff edge (shortest path); SARSA stays safer
# inland (because it "fears" its own ε-slips).
#
# Tiny comparison table:
#
#   Method       Target uses              Learns about          Typical vibe
#   -----------  -----------------------  --------------------  -------------
#   SARSA        Q(S', A') you will take  the exploring policy  cautious
#   Q-learning   max_a Q(S', a')          the greedy optimal    ambitious
#


# -----------------------------------------------------------------------------
# 4. DEEP Q-NETWORK (DQN) — Q-learning when the table does not fit
# -----------------------------------------------------------------------------
#
# Tabular Q: one number per (state, action). Fine for a 4×4 grid.
# Broken for Atari pixels or big continuous spaces — too many states.
#
# DQN idea: replace the table with a neural net Q(s, a; θ) that OUTPUTS
# action-values from a state (e.g. image → 4 joystick scores).
#
# Still the Q-learning target, but now a regression loss:
#   y = R + γ max_{a'} Q(S', a'; θ⁻)     (θ⁻ = a frozen "target network")
#   loss ≈ (y − Q(S, A; θ))²
#
# Two tricks that made deep RL actually work (Mnih et al., 2015):
#
#   (1) Experience replay
#       Store past transitions (S,A,R,S') in a big buffer. Train on RANDOM
#       mini-batches from the buffer — breaks the "correlated consecutive
#       frames" problem, like shuffling a dataset.
#
#   (2) Target network
#       Keep a slow-copy θ⁻ of the net for computing y. Update θ⁻ only
#       every N steps (or soft-update). Stops the moving-target chase
#       where the thing you chase is also the thing you train.
#
# Rough mental model:
#   Tabular Q-learning = flashcards for every room-door pair.
#   DQN               = a brain that looks at the room and guesses door scores.
#


# -----------------------------------------------------------------------------
# HOW THE PIECES FIT (roadmap for this notebook)
# -----------------------------------------------------------------------------
#
#   MC (previous)     wait for full return G          stable, slow, needs episodes
#   TD / SARSA        bootstrap with Q(S', A')        online, on-policy
#   Q-learning        bootstrap with max Q(S', ·)     online, off-policy
#   DQN               Q-learning + neural net         scales to big / pixel states
#
# Same goal as always: find a good policy π that maximizes discounted return.
# Different tools for when you get the learning signal (end vs each step) and
# how you represent Q (table vs network).
#
# Next cells: implement SARSA & Q-learning on GridWorld, compare them, then
# a small DQN sketch so the "table → network" jump feels concrete.
#


In [ ]:
# =============================================================================
# SETUP + GRIDWORLD — the tiny world SARSA / Q-learning / DQN will play in
# =============================================================================
#
# Same maze idea as Day 22 (copied here so this notebook runs alone):
#
#     (0,0) (0,1) (0,2) (0,3)
#     (1,0)  ##   (1,2) (1,3)      ## = wall
#     (2,0) (2,1)  ##   (2,3)
#     (3,0) (3,1) (3,2) [GOAL]
#
# Every step costs -1. Goal ends the episode. Short paths = higher return.
#
# For TD control we mostly need reset() / step() — play one move at a time.
# We keep P around for optional DP checks; SARSA/Q-learning will NOT read it.
#
# Error fixed below: `import gym` failed (package not installed, and we do not
# need it for GridWorld). Torch is optional here too — only required later
# for DQN. Tabular SARSA / Q-learning need only numpy.
#

import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict

# Seeds so re-runs look similar (still some randomness in exploration later)
np.random.seed(42)
random.seed(42)

# Torch is for the DQN cell later — import softly so tabular cells still run
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.nn.functional as F

    torch.manual_seed(42)
    TORCH_OK = True
    print(f"PyTorch {torch.__version__} ready (for DQN later).")
except ImportError:
    TORCH_OK = False
    print("PyTorch not installed — tabular SARSA/Q-learning still work.")

# Classic `gym` is optional / often missing. Prefer gymnasium if present.
# GridWorld does not use either; only a future Atari-style DQN demo would.
try:
    import gymnasium as gym  # modern fork

    GYM_OK = True
    print("gymnasium available (optional).")
except ImportError:
    try:
        import gym  # older package name

        GYM_OK = True
        print("gym available (optional).")
    except ImportError:
        GYM_OK = False
        print("No gym/gymnasium — fine for this notebook's GridWorld demos.")


class GridWorld:
    """
    Tiny grid MDP with a Gym-style API:

      state  = (row, col)
      action = 'up' | 'down' | 'left' | 'right'
      step   → (next_state, reward, done, info)

    Bounce off walls/obstacles (stay put, still pay -1).
    """

    def __init__(self, size=4, obstacles=None, terminal=None, gamma=0.9, slip_prob=0.0, cliffs=None, cliff_penalty=-100.0, fixed_start=None):
        self.size = size
        # Every cell is a potential state (hashable tuple → good dict key for Q)
        self.states = [(i, j) for i in range(size) for j in range(size)]
        self.terminal = terminal if terminal is not None else (size - 1, size - 1)
        self.obstacles = list(obstacles) if obstacles is not None else [(1, 1), (2, 2)]

        self.actions = ["up", "down", "left", "right"]
        # How each action nudges (row, col). "up" decreases row (toward top).
        self.action_effects = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1),
        }

        self.gamma = gamma
        self.reward_step = -1.0  # living cost → prefer fewer steps to goal
        self.reward_terminal = 0.0
        # slip_prob: with this chance, step() picks a random action (ice/wind).
        self.slip_prob = float(slip_prob)
        # cliffs: cells that give cliff_penalty and end the episode (Sutton cliff).
        self.cliffs = set(cliffs or [])
        self.cliff_penalty = float(cliff_penalty)
        self.fixed_start = fixed_start  # if set, reset() always starts here

        # Optional model P[s][a] = [(prob, next_s, reward, done), ...]
        # With slip, P is a mixture (same rule as step). TD methods use step().
        self.P = {}
        nA = len(self.actions)
        for s in self.states:
            if s == self.terminal or s in self.obstacles or s in self.cliffs:
                continue
            self.P[s] = {}
            for a in self.actions:
                mass = {}  # next_s -> prob
                for a_act in self.actions:
                    p = ((1.0 - self.slip_prob) + self.slip_prob / nA) if a_act == a else (self.slip_prob / nA)
                    ns = self._get_next_state(s, a_act)
                    mass[ns] = mass.get(ns, 0.0) + p
                transitions = []
                for ns, p in mass.items():
                    if p <= 1e-12:
                        continue
                    if ns in self.cliffs:
                        transitions.append((p, ns, self.cliff_penalty, True))
                    else:
                        transitions.append((p, ns, self.reward_step, ns == self.terminal))
                self.P[s][a] = transitions

    def _get_next_state(self, state, action):
        """Physics of one move. Illegal → stay in place."""
        if state == self.terminal or state in self.obstacles:
            return state
        i, j = state
        di, dj = self.action_effects[action]
        ni, nj = i + di, j + dj
        if 0 <= ni < self.size and 0 <= nj < self.size and (ni, nj) not in self.obstacles:
            return (ni, nj)
        return state  # hit wall / obstacle → bounce

    def reset(self):
        """Start a new episode from a random free cell (or fixed_start)."""
        if self.fixed_start is not None:
            self.current_state = self.fixed_start
            return self.current_state
        available = [
            s
            for s in self.states
            if s != self.terminal and s not in self.obstacles and s not in self.cliffs
        ]
        self.current_state = random.choice(available)
        return self.current_state

    def step(self, action):
        """
        Take one action. Returns Gym-style tuple:
          next_state, reward, done, info

        If slip_prob > 0, sometimes a random action runs instead.
        Landing on a cliff → cliff_penalty and done=True.
        """
        if self.slip_prob > 0.0 and random.random() < self.slip_prob:
            action = random.choice(self.actions)
        next_s = self._get_next_state(self.current_state, action)
        if next_s in self.cliffs:
            reward = self.cliff_penalty
            done = True
        else:
            reward = self.reward_step
            done = next_s == self.terminal
        self.current_state = next_s
        return next_s, reward, done, {}


# --- sanity check: env works without gym ------------------------------------
env = GridWorld(size=4)
print(f"\nGridWorld size={env.size}  states={len(env.states)}  free={len(env.P)}")
print(f"Terminal={env.terminal}  Obstacles={env.obstacles}")
print("Example P[(0,0)]['right']:", env.P[(0, 0)]["right"])

s = env.reset()
print(f"Episode demo — start {s}")
for _ in range(4):
    a = random.choice(env.actions)
    s, r, done, _ = env.step(a)
    print(f"  {a:5s} → {s}  reward={r}  done={done}")
    if done:
        break

# Picture of the board
board = np.zeros((env.size, env.size))
for oi, oj in env.obstacles:
    board[oi, oj] = -1
board[env.terminal] = 2
plt.figure(figsize=(4, 4))
plt.imshow(board, cmap="RdYlGn", vmin=-1, vmax=2)
for i in range(env.size):
    for j in range(env.size):
        if (i, j) in env.obstacles:
            plt.text(j, i, "X", ha="center", va="center", color="white", fontsize=14)
        elif (i, j) == env.terminal:
            plt.text(j, i, "G", ha="center", va="center", fontweight="bold", fontsize=14)
        else:
            plt.text(j, i, f"{i},{j}", ha="center", va="center", color="gray", fontsize=8)
plt.title("GridWorld — green=goal, red=wall")
plt.axis("off")
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# reset/step  → what SARSA & Q-learning call every episode (model-free play).
# P           → optional rulebook; not required for TD (unlike DP).
# reward -1   → optimal policy reaches G in as few steps as possible.
# TORCH_OK / GYM_OK → flags for later cells; missing gym no longer crashes setup.
# Extra knobs: slip_prob (random slips), cliffs=... (dangerous cells).
#


In [ ]:
# =============================================================================
# TABULAR Q-LEARNING — off-policy TD control (dream of the BEST next move)
# =============================================================================
#
# Remember from the intro:
#   Q(S,A) ← Q(S,A) + α [ R + γ max_{a'} Q(S', a')  −  Q(S,A) ]
#                              └──────────┬──────────┘
#                         bootstrap with the BEST action next
#
# Plain English loop each step:
#   1. Pick action with ε-greedy (sometimes random = explore)
#   2. Take it, see reward R and next state S'
#   3. Pretend next action is argmax Q(S', ·) — even if we won't do that
#   4. Nudge Q(S,A) toward  R + γ · that best next value
#
# Off-policy: we LEARN the greedy optimal policy while we BEHAVE with
# exploration. Bold / ambitious personality.
#
# Q table layout here:
#   Q[state] is a length-4 array → values for [up, down, left, right]
#


def epsilon_greedy(Q, state, actions, epsilon):
    """With prob ε pick random action; else pick argmax Q[state]."""
    if random.random() < epsilon:
        return random.randrange(len(actions))
    return int(np.argmax(Q[state]))


def q_learning(
    env,
    num_episodes=500,
    alpha=0.1,
    gamma=0.9,
    epsilon=0.1,
    epsilon_decay=0.995,
    min_epsilon=0.01,
):
    """
    Learn Q* by playing. Returns (Q, greedy_policy, episode_rewards).

    alpha   = learning rate (how hard we yank Q toward the TD target)
    gamma   = discount (same γ as the MDP)
    epsilon = explore rate; decays each episode toward min_epsilon
    """
    # Every new state starts with Q=0 for all actions ("I know nothing yet")
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    episode_rewards = []

    for ep in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            # --- behave: ε-greedy (explore OR exploit) -----------------------
            action_idx = epsilon_greedy(Q, state, env.actions, epsilon)
            action = env.actions[action_idx]
            next_state, reward, done, _ = env.step(action)

            # --- learn: Q-learning target uses max over NEXT actions ---------
            # At the goal there is no future → bootstrap value is 0
            if done or next_state == env.terminal:
                best_next = 0.0
            else:
                best_next = float(np.max(Q[next_state]))

            td_target = reward + gamma * best_next
            td_error = td_target - Q[state][action_idx]
            Q[state][action_idx] += alpha * td_error  # move a little toward target

            state = next_state
            total_reward += reward

        episode_rewards.append(total_reward)
        # Slowly explore less as we (hopefully) know more
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

    # Greedy policy for plotting / evaluation (no ε)
    policy = {}
    for s in env.states:
        if s != env.terminal and s not in env.obstacles:
            policy[s] = env.actions[int(np.argmax(Q[s]))]
    return Q, policy, episode_rewards


# --- run + peek -------------------------------------------------------------
# Ensure env exists even if cells were run out of order / kernel restarted
try:
    env
except NameError:
    env = GridWorld(size=4)
    print("Note: created default env = GridWorld(size=4)")

Q_q, pi_q, rewards_q = q_learning(env, num_episodes=800)
print("Q-learning done. Sample greedy policy:")
for s in sorted(pi_q)[:6]:
    print(f"  {s} → {pi_q[s]:5s}  Q={np.round(Q_q[s], 2)}")
print(f"Mean return (last 50 eps): {np.mean(rewards_q[-50:]):.2f}")

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# td_target = R + γ max Q(S')     ← the "dream best" next move
# td_error  = target − current Q  ← surprise (positive = better than expected)
# epsilon decay: start curious, end mostly greedy
#


In [ ]:
# =============================================================================
# TABULAR SARSA — on-policy TD control (learn from the move you WILL take)
# =============================================================================
#
# Name = the five-tuple used each update:  (S, A, R, S', A')
#
# Update:
#   Q(S,A) ← Q(S,A) + α [ R + γ Q(S', A')  −  Q(S,A) ]
#                              └─────┬─────┘
#                         the NEXT action A' from the SAME ε-greedy policy
#
# vs Q-learning:
#   Q-learning target → γ · max_a Q(S', a)     (best possible next)
#   SARSA target      → γ · Q(S', A')          (next action we actually sample)
#
# On-policy: learn about the policy you are following (including exploration).
# Cautious personality — if ε sometimes slips near danger, SARSA "feels" that.
#


def sarsa(
    env,
    num_episodes=500,
    alpha=0.1,
    gamma=0.9,
    epsilon=0.1,
    epsilon_decay=0.995,
    min_epsilon=0.01,
):
    """
    Same API as q_learning: returns (Q, greedy_policy, episode_rewards).

    Key difference inside the loop: choose A' BEFORE the update, and plug
    Q(S', A') into the target (not max_a Q).
    """
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    episode_rewards = []

    for ep in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0.0

        # SARSA needs an action in hand BEFORE the first step (the "A" in S,A,…)
        action_idx = epsilon_greedy(Q, state, env.actions, epsilon)

        while not done:
            action = env.actions[action_idx]
            next_state, reward, done, _ = env.step(action)

            # Choose A' with the SAME ε-greedy rule (on-policy)
            if done or next_state == env.terminal:
                # No next action at the goal — bootstrap with 0
                next_action_idx = 0
                next_q = 0.0
            else:
                next_action_idx = epsilon_greedy(Q, next_state, env.actions, epsilon)
                next_q = float(Q[next_state][next_action_idx])

            td_target = reward + gamma * next_q
            td_error = td_target - Q[state][action_idx]
            Q[state][action_idx] += alpha * td_error

            # Slide the window: (S', A') becomes the new (S, A)
            state = next_state
            action_idx = next_action_idx
            total_reward += reward

        episode_rewards.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

    policy = {}
    for s in env.states:
        if s != env.terminal and s not in env.obstacles:
            policy[s] = env.actions[int(np.argmax(Q[s]))]
    return Q, policy, episode_rewards


# --- run SARSA and compare to Q-learning ------------------------------------
# Ensure helpers + env exist even if you skipped / reordered cells
try:
    epsilon_greedy
except NameError:
    def epsilon_greedy(Q, state, actions, epsilon):
        if random.random() < epsilon:
            return random.randrange(len(actions))
        return int(np.argmax(Q[state]))

try:
    env
except NameError:
    try:
        env = GridWorld(size=4)
    except NameError as e:
        raise RuntimeError(
            "GridWorld is missing. Run the Setup/GridWorld cell once, then re-run this cell."
        ) from e
    print("Note: created default env = GridWorld(size=4)")

Q_s, pi_s, rewards_s = sarsa(env, num_episodes=800)
print("SARSA done. Sample greedy policy:")
for s in sorted(pi_s)[:6]:
    print(f"  {s} → {pi_s[s]:5s}  Q={np.round(Q_s[s], 2)}")
print(f"Mean return (last 50 eps): {np.mean(rewards_s[-50:]):.2f}")

# Learning curves side by side (smoothed a bit)
def smooth(x, w=25):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

plt.figure(figsize=(8, 4))
if "rewards_q" in globals():
    plt.plot(smooth(rewards_q), label="Q-learning", alpha=0.9)
plt.plot(smooth(rewards_s), label="SARSA", alpha=0.9)
plt.xlabel("episode (smoothed)")
plt.ylabel("return")
plt.title("Both climb toward shorter paths (less negative return)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Policy arrows for both (if Q-learning already ran)
arrow = {"up": "↑", "down": "↓", "left": "←", "right": "→"}

def show_policy(policy, title):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_xlim(-0.5, env.size - 0.5)
    ax.set_ylim(env.size - 0.5, -0.5)
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
    ax.grid(True)
    for oi, oj in env.obstacles:
        ax.add_patch(plt.Rectangle((oj - 0.5, oi - 0.5), 1, 1, color="gray"))
    ti, tj = env.terminal
    ax.add_patch(plt.Rectangle((tj - 0.5, ti - 0.5), 1, 1, color="green"))
    for (i, j), a in policy.items():
        ax.text(j, i, arrow[a], ha="center", va="center", fontsize=18)
    ax.set_title(title)
    plt.show()

show_policy(pi_s, "SARSA greedy π")
if "pi_q" in globals():
    show_policy(pi_q, "Q-learning greedy π")
    agree = sum(pi_q[s] == pi_s[s] for s in pi_q) / len(pi_q)
    print(f"Policy agreement (Q vs SARSA): {100 * agree:.0f}%")

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# Q-learning: target uses max Q(S')     → learns optimal greedy path
# SARSA:      target uses Q(S', A')     → learns about ε-greedy behavior
# On this safe grid they often look similar; on a "cliff" they'd diverge.
#


In [ ]:
# =============================================================================
# COMPARE — Q-learning vs SARSA vs exact DP on the same GridWorld
# =============================================================================
#
# Idea in one sentence:
#   DP (value iteration) uses the rulebook P → exact π* (for this tiny MDP).
#   Q-learning / SARSA learn from play → approximate π, noisy learning curves.
#
# What to look for:
#   • Policies should mostly MATCH the DP arrows (same shortest-path idea).
#   • Returns start very negative (wandering) and climb toward ~-3 to -5.
#   • Q-learning vs SARSA often look similar on this SAFE grid; they diverge
#     more on risky maps (classic "cliff walking").
#

# Need GridWorld + q_learning + sarsa from earlier cells
missing = [n for n in ("GridWorld", "q_learning", "sarsa") if n not in globals()]
if missing:
    raise RuntimeError(
        "Missing: " + ", ".join(missing) + ". Run the Setup, Q-learning, and SARSA cells first."
    )

env = GridWorld(size=4)

Q_ql, policy_ql, rewards_ql = q_learning(env, num_episodes=1000, alpha=0.1, epsilon=0.1)
Q_sarsa, policy_sarsa, rewards_sarsa = sarsa(env, num_episodes=1000, alpha=0.1, epsilon=0.1)


def value_iteration(env, theta=1e-6):
    """
    Exact planning with env.P (from Day 22).
    V(s) ← max_a Σ P(s'|s,a)[ R + γ V(s') ]  until V stops changing.
    """
    V = {s: 0.0 for s in env.states if s not in env.obstacles}
    while True:
        delta = 0.0
        for s in V:
            if s == env.terminal:
                V[s] = 0.0
                continue
            v_old = V[s]
            best = -float("inf")
            for a in env.actions:
                q = sum(
                    p * (r + env.gamma * V[next_s])
                    for p, next_s, r, done in env.P[s][a]
                )
                best = max(best, q)
            V[s] = best
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    policy = {}
    for s in env.P:
        q_vals = {
            a: sum(p * (r + env.gamma * V[next_s]) for p, next_s, r, done in env.P[s][a])
            for a in env.actions
        }
        policy[s] = max(q_vals, key=q_vals.get)
    return policy, V


opt_policy, opt_V = value_iteration(env)

def agreement(pi_a, pi_b):
    keys = sorted(set(pi_a) & set(pi_b))
    return sum(pi_a[s] == pi_b[s] for s in keys) / max(len(keys), 1)

print("Optimal DP policy:", opt_policy)
print("Q-learning policy:", policy_ql)
print("SARSA policy:     ", policy_sarsa)
print(f"Agree with DP — Q-learning: {100 * agreement(policy_ql, opt_policy):.0f}%")
print(f"Agree with DP — SARSA:      {100 * agreement(policy_sarsa, opt_policy):.0f}%")

# Smoothed learning curves (raw rewards are very noisy)
def smooth(x, w=30):
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode="valid") if len(x) >= w else x

plt.figure(figsize=(10, 5))
plt.plot(smooth(rewards_ql), label="Q-learning")
plt.plot(smooth(rewards_sarsa), label="SARSA")
plt.xlabel("Episode (smoothed)")
plt.ylabel("Total reward (higher = better / shorter path)")
plt.legend()
plt.title("Learning curves on GridWorld")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# DP policy = ground truth for this known tiny MDP.
# TD policies ≈ DP when enough episodes; not bit-identical (exploration noise).
# Curve going UP (less negative) = agent reaches the goal faster on average.
#


In [ ]:
# =============================================================================
# DEEP Q-NETWORK (DQN) on CartPole — Q-learning when a TABLE cannot fit
# =============================================================================
#
# CartPole (gymnasium): balance a pole on a moving cart.
#   State  = 4 numbers (cart x, cart velocity, pole angle, pole angular vel)
#            → CONTINUOUS → infinite "cells" → cannot use tabular Q[s][a]
#   Actions = 0 (push left) or 1 (push right)
#   Reward  = +1 every timestep the pole stays up (episode ends if it falls)
#
# DQN = Q-learning idea + neural net instead of a table:
#   Q(s, a; θ) ≈ network that scores each action from state s
#
# Two tricks that keep training stable:
#   (1) Replay buffer — store past (s,a,r,s',done), train on random batches
#       (breaks correlation of consecutive frames; like shuffling a dataset)
#   (2) Target network — slow copy θ⁻ used only to build TD targets
#       y = r + γ max_a' Q(s', a'; θ⁻)    (or just r if done)
#
# This cell is a small educational DQN (not a competition solver). Expect the
# return curve to trend UP over a few hundred episodes on CPU.
#

from collections import deque

if not globals().get("TORCH_OK", False):
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        import torch.nn.functional as F
        TORCH_OK = True
    except ImportError as e:
        raise RuntimeError("PyTorch required for DQN. Install torch in the venv.") from e

try:
    import gymnasium as gym
except ImportError as e:
    raise RuntimeError(
        "Install gymnasium:  pip install gymnasium"
    ) from e


class ReplayBuffer:
    """Circular bag of past transitions. sample() = random mini-batch."""

    def __init__(self, capacity=10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            np.array(s, dtype=np.float32),
            np.array(a, dtype=np.int64),
            np.array(r, dtype=np.float32),
            np.array(s2, dtype=np.float32),
            np.array(d, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buf)


class QNet(nn.Module):
    """Tiny MLP: state (4,) → Q-values for each action (2,)."""

    def __init__(self, n_obs, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


def dqn_cartpole(
    num_episodes=400,
    gamma=0.99,
    lr=1e-3,
    batch_size=64,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    target_update_every=10,
    max_steps=500,
    hidden=128,
    use_replay=True,
    use_target=True,
    quiet=False,
):
    env = gym.make("CartPole-v1")
    n_obs = env.observation_space.shape[0]
    n_actions = env.action_space.n

    policy_net = QNet(n_obs, n_actions, hidden=hidden)   # θ  — trained every step
    target_net = QNet(n_obs, n_actions, hidden=hidden)   # θ⁻ — frozen copy for targets
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    buffer = ReplayBuffer(20_000)
    epsilon = epsilon_start
    returns = []

    for ep in range(num_episodes):
        obs, _ = env.reset(seed=ep)
        done = False
        truncated = False
        total_r = 0.0
        steps = 0

        while not (done or truncated) and steps < max_steps:
            # ε-greedy over network Q-values
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    q = policy_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
                    action = int(q.argmax(dim=1).item())

            obs_before = obs
            next_obs, reward, done, truncated, _ = env.step(action)
            buffer.push(obs_before, action, reward, next_obs, float(done or truncated))
            obs = next_obs
            total_r += reward
            steps += 1

            # Learn: replay mini-batch OR (ablation) the single latest transition
            if use_replay:
                if len(buffer) < batch_size:
                    continue
                s, a, r, s2, d = buffer.sample(batch_size)
            else:
                s = np.array([obs_before], dtype=np.float32)
                a = np.array([action], dtype=np.int64)
                r = np.array([reward], dtype=np.float32)
                s2 = np.array([next_obs], dtype=np.float32)
                d = np.array([float(done or truncated)], dtype=np.float32)

            s_t = torch.tensor(s)
            a_t = torch.tensor(a).unsqueeze(1)
            r_t = torch.tensor(r)
            s2_t = torch.tensor(s2)
            d_t = torch.tensor(d)

            q_sa = policy_net(s_t).gather(1, a_t).squeeze(1)
            with torch.no_grad():
                net_for_target = target_net if use_target else policy_net
                max_next = net_for_target(s2_t).max(dim=1).values
                y = r_t + gamma * max_next * (1.0 - d_t)

            loss = F.mse_loss(q_sa, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        returns.append(total_r)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        # Periodically copy θ → θ⁻ (skip when ablating target network)
        if use_target and (ep + 1) % target_update_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if not quiet and (ep + 1) % 50 == 0:
            print(
                f"ep {ep+1:4d}  return={total_r:6.1f}  "
                f"avg50={np.mean(returns[-50:]):6.1f}  eps={epsilon:.2f}"
            )

    env.close()
    return returns, policy_net


returns_dqn, dqn_net = dqn_cartpole(num_episodes=400)

def smooth(x, w=20):
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode="valid") if len(x) >= w else x

plt.figure(figsize=(9, 4))
plt.plot(returns_dqn, alpha=0.25, label="raw")
plt.plot(smooth(returns_dqn), label="smoothed")
plt.axhline(195, color="green", linestyle="--", alpha=0.6, label="solved≈195")
plt.xlabel("Episode")
plt.ylabel("Return (timesteps balanced)")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final avg return (last 50): {np.mean(returns_dqn[-50:]):.1f}")

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# Tabular Q: flashcards for every discrete state.
# DQN:        a small brain scores actions from continuous features.
# Replay:     learn from old memories, not only the last step.
# Target net: keep the TD target from chasing a moving finish line.
# Higher return = pole stayed up longer (max 500 on CartPole-v1).
#


In [ ]:
# =============================================================================
# MINI-PROJECT — stochastic SARSA vs Q, DQN tuning, Q heatmaps, portfolio
# =============================================================================
#
# Prereqs: run Setup, Q-learning, SARSA, and DQN cells first (defs must exist).
#
from pathlib import Path

ARTIFACTS = Path("week4/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

missing = [
    n
    for n in ("GridWorld", "q_learning", "sarsa", "dqn_cartpole", "QNet")
    if n not in globals()
]
if missing:
    raise RuntimeError("Missing " + ", ".join(missing) + " — run earlier cells first.")


def smooth(x, w=25):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")


def policy_arrows(env, policy, title):
    arrow = {"up": "↑", "down": "↓", "left": "←", "right": "→"}
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.set_xlim(-0.5, env.size - 0.5)
    ax.set_ylim(env.size - 0.5, -0.5)
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
    ax.grid(True)
    for oi, oj in env.obstacles:
        ax.add_patch(plt.Rectangle((oj - 0.5, oi - 0.5), 1, 1, color="gray"))
    for ci, cj in getattr(env, "cliffs", []):
        ax.add_patch(plt.Rectangle((cj - 0.5, ci - 0.5), 1, 1, color="black"))
    ti, tj = env.terminal
    ax.add_patch(plt.Rectangle((tj - 0.5, ti - 0.5), 1, 1, color="green"))
    for (i, j), a in policy.items():
        ax.text(j, i, arrow[a], ha="center", va="center", fontsize=16)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# =============================================================================
# 3.1  Stochastic transitions (+ cliff for "safer policy" intuition)
# =============================================================================
print("=" * 64)
print("3.1  Q-learning vs SARSA with slip (and a cliff world)")
print("=" * 64)

# --- A) same 4×4 maze, 10% random action ------------------------------------
env_slip = GridWorld(size=4, slip_prob=0.1)
Q_q_slip, pi_q_slip, rew_q_slip = q_learning(
    env_slip, num_episodes=2000, alpha=0.1, epsilon=0.15, gamma=0.9
)
Q_s_slip, pi_s_slip, rew_s_slip = sarsa(
    env_slip, num_episodes=2000, alpha=0.1, epsilon=0.15, gamma=0.9
)

agree = sum(pi_q_slip[s] == pi_s_slip[s] for s in pi_q_slip) / len(pi_q_slip)
print(f"Slip grid — policy agreement Q vs SARSA: {100 * agree:.0f}%")
print(f"Mean return last 100 — Q: {np.mean(rew_q_slip[-100:]):.2f}  "
      f"SARSA: {np.mean(rew_s_slip[-100:]):.2f}")

plt.figure(figsize=(9, 4))
plt.plot(smooth(rew_q_slip), label="Q-learning (slip=0.1)")
plt.plot(smooth(rew_s_slip), label="SARSA (slip=0.1)")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.grid(True, alpha=0.3)
plt.title("3.1a Slippery GridWorld — both cope; curves stay noisy")
plt.tight_layout(); plt.show()

# --- B) Cliff world: short path hugs cliff; safe path walks inland -----------
# Start S=(3,0), goal G=(3,3), cliffs on the bottom between them.
# Q-learning often hugs the cliff (optimal if greedy). SARSA stays safer
# inland because ε-greedy might fall IN while learning.
env_cliff = GridWorld(
    size=4,
    obstacles=[],
    terminal=(3, 3),
    cliffs=[(3, 1), (3, 2)],
    cliff_penalty=-100.0,
    fixed_start=(3, 0),
    slip_prob=0.0,
)

Q_q_c, pi_q_c, rew_q_c = q_learning(
    env_cliff, num_episodes=3000, alpha=0.5, epsilon=0.1, gamma=1.0, epsilon_decay=1.0, min_epsilon=0.1
)
Q_s_c, pi_s_c, rew_s_c = sarsa(
    env_cliff, num_episodes=3000, alpha=0.5, epsilon=0.1, gamma=1.0, epsilon_decay=1.0, min_epsilon=0.1
)

print("\nCliff world policies (black=cliff, green=goal). Safe path goes UP then across.")
print("Q-learning π:", pi_q_c)
print("SARSA π:     ", pi_s_c)
print(f"Mean return last 200 — Q: {np.mean(rew_q_c[-200:]):.1f}  SARSA: {np.mean(rew_s_c[-200:]):.1f}")
print("(During training with ε>0, SARSA usually gets HIGHER avg return = fewer cliff falls.)")

policy_arrows(env_cliff, pi_q_c, "3.1b Q-learning π (often hugs cliff)")
policy_arrows(env_cliff, pi_s_c, "3.1b SARSA π (often safer / inland)")

plt.figure(figsize=(9, 4))
plt.plot(smooth(rew_q_c, 50), label="Q-learning")
plt.plot(smooth(rew_s_c, 50), label="SARSA")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.grid(True, alpha=0.3)
plt.title("3.1b Cliff world — SARSA's on-policy updates punish near-cliff exploration")
plt.tight_layout(); plt.show()

print(
    """
TAKEAWAY 3.1
  Slip alone: both algorithms adapt; policies often similar.
  Cliff + ε-greedy: Q-learning learns the optimal *greedy* path (along the edge).
  SARSA learns about the *exploring* policy → stays away from the cliff (safer).
"""
)


# =============================================================================
# 3.3  Visualize tabular Q-values (do this before long DQN sweeps)
# =============================================================================
print("=" * 64)
print("3.3  Q-value heatmaps (tabular Q-learning on deterministic 4×4)")
print("=" * 64)

env_viz = GridWorld(size=4, slip_prob=0.0)
Q_viz, pi_viz, _ = q_learning(env_viz, num_episodes=1500, alpha=0.1, epsilon=0.1)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, a_idx, a_name in zip(axes, range(4), env_viz.actions):
    grid = np.full((env_viz.size, env_viz.size), np.nan)
    for s in env_viz.states:
        if s in env_viz.obstacles or s == env_viz.terminal:
            continue
        grid[s] = Q_viz[s][a_idx]
    im = ax.imshow(grid, cmap="viridis")
    for oi, oj in env_viz.obstacles:
        ax.text(oj, oi, "X", ha="center", va="center", color="white")
    ax.text(*env_viz.terminal[::-1], "G", ha="center", va="center", color="white", fontweight="bold")
    # mark greedy action with a white border cell text
    for s, a in pi_viz.items():
        if a == a_name:
            ax.text(s[1], s[0], "★", ha="center", va="center", color="white", fontsize=12)
    ax.set_title(f"Q(:, {a_name})")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("3.3 Brighter = better action-value; ★ = greedy action in that cell")
plt.tight_layout(); plt.show()


# =============================================================================
# 3.2  Tune DQN hyperparameters (+ ablations)
# =============================================================================
print("=" * 64)
print("3.2  DQN hyperparameter sweep + ablations (CartPole, short runs)")
print("=" * 64)

configs = [
    {"name": "baseline", "lr": 1e-3, "epsilon_decay": 0.995, "target_update_every": 10, "hidden": 128},
    {"name": "lr_high", "lr": 5e-3, "epsilon_decay": 0.995, "target_update_every": 10, "hidden": 128},
    {"name": "lr_low", "lr": 1e-4, "epsilon_decay": 0.995, "target_update_every": 10, "hidden": 128},
    {"name": "slow_eps", "lr": 1e-3, "epsilon_decay": 0.99, "target_update_every": 10, "hidden": 128},
    {"name": "fast_target", "lr": 1e-3, "epsilon_decay": 0.995, "target_update_every": 2, "hidden": 128},
    {"name": "wide_net", "lr": 1e-3, "epsilon_decay": 0.995, "target_update_every": 10, "hidden": 256},
]

sweep_results = []
plt.figure(figsize=(10, 5))
for cfg in configs:
    name = cfg["name"]
    kw = {k: v for k, v in cfg.items() if k != "name"}
    print(f"  training {name} ...")
    rets, _ = dqn_cartpole(num_episodes=180, quiet=True, **kw)
    avg = float(np.mean(rets[-40:]))
    std = float(np.std(rets[-40:]))
    sweep_results.append((name, avg, std, rets))
    plt.plot(smooth(rets, 15), label=f"{name} (avg40={avg:.0f})")
    print(f"    → avg last 40 return = {avg:.1f} ± {std:.1f}")

plt.xlabel("episode"); plt.ylabel("return"); plt.title("3.2 Hyperparameter sweep")
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f"\n{'name':12s} {'avg40':>8s} {'std40':>8s}")
for name, avg, std, _ in sorted(sweep_results, key=lambda t: -t[1]):
    print(f"{name:12s} {avg:8.1f} {std:8.1f}")

print("\nAblations: remove target network / remove replay buffer")
ablations = [
    ("full DQN", dict(use_replay=True, use_target=True)),
    ("no target", dict(use_replay=True, use_target=False)),
    ("no replay", dict(use_replay=False, use_target=True)),
]
plt.figure(figsize=(10, 4))
ablation_rows = []
for name, kw in ablations:
    print(f"  training {name} ...")
    rets, _ = dqn_cartpole(num_episodes=180, quiet=True, **kw)
    avg = float(np.mean(rets[-40:]))
    ablation_rows.append((name, avg, rets))
    plt.plot(smooth(rets, 15), label=f"{name} (avg40={avg:.0f})")
    print(f"    → avg last 40 = {avg:.1f}")
plt.legend(); plt.title("3.2 Ablations — target net & replay stabilize learning")
plt.xlabel("episode"); plt.ylabel("return"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


# =============================================================================
# 3.4  Portfolio artifact — discussion, longer DQN train, save weights
# =============================================================================
print("=" * 64)
print("3.4  Portfolio: final DQN train + save + short write-up")
print("=" * 64)

returns_final, dqn_final = dqn_cartpole(num_episodes=400, quiet=False)
model_path = ARTIFACTS / "dqn_cartpole.pt"
torch.save(
    {
        "state_dict": dqn_final.state_dict(),
        "hidden": 128,
        "n_obs": 4,
        "n_actions": 2,
        "avg_return_last_50": float(np.mean(returns_final[-50:])),
    },
    model_path,
)
print(f"Saved trained DQN → {model_path.resolve()}")

# Persist a small results summary for the portfolio
summary_path = ARTIFACTS / "day23_results_summary.txt"
with open(summary_path, "w") as f:
    f.write("Day 23 mini-project summary\n")
    f.write("===========================\n\n")
    f.write("3.1 Slip agreement Q vs SARSA: "
            f"{100 * agree:.0f}%\n")
    f.write(f"3.1 Cliff mean return Q={np.mean(rew_q_c[-200:]):.1f} "
            f"SARSA={np.mean(rew_s_c[-200:]):.1f}\n\n")
    f.write("3.2 Hyperparameter avg40 returns:\n")
    for name, avg, std, _ in sweep_results:
        f.write(f"  {name}: {avg:.1f} ± {std:.1f}\n")
    f.write("\n3.2 Ablations avg40:\n")
    for name, avg, _ in ablation_rows:
        f.write(f"  {name}: {avg:.1f}\n")
    f.write(f"\n3.4 Final DQN avg50: {np.mean(returns_final[-50:]):.1f}\n")
    f.write(f"Model: {model_path}\n")
print(f"Wrote summary → {summary_path}")

plt.figure(figsize=(9, 4))
plt.plot(returns_final, alpha=0.25)
plt.plot(smooth(returns_final), label="smoothed")
plt.axhline(195, color="green", ls="--", alpha=0.5, label="solved≈195")
plt.title("3.4 Final DQN learning curve (CartPole)")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(
    """
================================================================================
DISCUSSION (portfolio write-up — also in comments for GitHub notebook)
================================================================================

On-policy vs off-policy
  • SARSA (on-policy): TD target uses the next action A' you WILL take under
    ε-greedy. It "fears" its own exploration → safer policies near danger
    (cliff demo).
  • Q-learning (off-policy): TD target uses max_a Q(S',a) — the best action —
    even while behaving with exploration. Learns the optimal greedy path,
    which can hug risk if exploration is turned off at test time.

Experience replay
  • Stores (s,a,r,s') and trains on random mini-batches.
  • Breaks temporal correlation (consecutive frames are highly similar).
  • Lets the agent reuse rare / valuable transitions many times.
  • Ablation "no replay" usually looks less stable / lower return.

Target network
  • Frozen copy θ⁻ builds the TD target y = r + γ max Q(s'; θ⁻).
  • Stops the chase where the target moves every gradient step.
  • Ablation "no target" often oscillates or plateaus earlier.

Hyperparameters (from this run's sweep)
  • Too-large lr → noisy / unstable; too-small → slow learning.
  • Slower ε decay → more exploration for longer (can help early, hurt late).
  • Target update frequency / width trade stability vs freshness.

This notebook already contains tabular Q/SARSA, DP comparison, DQN on CartPole,
learning curves, and the experiments above — push it with week4/artifacts/.
"""
)
